# 🦶 Flatfoot (Pes Planus) Classification with Custom FootArchNet
### Biomechanically-Guided Multi-Scale Attention Network on Google Colab

This notebook trains and evaluates the custom **FootArchNet** model for automated classification of Flatfoot (*Pes Planus*) vs. Normal Foot using weight-bearing lateral foot radiographs.

---
### 📋 Pipeline Highlights:
1. **Custom Architecture (`FootArchNet`):** Combines anisotropic Strip Pooling, Biomechanical Attention, and Multi-Scale Feature Fusion specifically engineered for longitudinal arch collapse.
2. **GPU Acceleration:** Automatic Mixed Precision (AMP / FP16) training on Google Colab GPU (NVIDIA T4 / V100 / A100).
3. **Rigorous Evaluation:** Evaluated on the held-out 230-patient test cohort and benchmarked against standard ResNet-50 and EfficientNet-B2 baselines.
4. **Explainable AI (Grad-CAM):** Anatomical saliency mapping confirming focus on the medial longitudinal arch, navicular-cuneiform sag, and calcaneal pitch.



## 1. Hardware Verification & GPU Setup
Check that Google Colab GPU accelerator is enabled (Runtime > Change runtime type > GPU).


In [ ]:
import torch
import os
import sys

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    !nvidia-smi
else:
    print("⚠️ WARNING: Running on CPU! Please go to Runtime > Change runtime type > GPU to enable hardware acceleration.")



## 2. Codebase Setup
Clone the private repository or copy project files into Colab.


In [ ]:
# If running directly inside cloned repo:
if os.path.exists("src/models/foot_arch_net.py"):
    print("✓ Already inside flatfoot_classifier repository directory.")
else:
    # Clone repository (replace with your personal access token if needed for private repo)
    print("Setting up repository...")
    # Option A: Public or authenticated token clone
    # !git clone https://<TOKEN>@github.com/tahsinsoyak/flatfoot_classifier.git
    # %cd flatfoot_classifier

import sys
from pathlib import Path
repo_root = Path(".").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print(f"Project root added to sys.path: {repo_root}")



## 3. Install Required Dependencies


In [ ]:
!pip install -q albumentations timm scikit-learn seaborn matplotlib pandas pillow tqdm
print("✓ All dependencies installed successfully.")



## 4. Dataset Ingestion (Google Drive or Upload)
Mount Google Drive to load the compact `flatfoot_processed_dataset.zip` (~90 MB) created by `scripts/package_dataset_for_colab.py`.



In [ ]:
import zipfile
from pathlib import Path

zip_candidates = [
    Path("/content/drive/MyDrive/flatfoot_processed_dataset.zip"),
    Path("/content/drive/MyDrive/flatfoot_classifier/flatfoot_processed_dataset.zip"),
    Path("data/flatfoot_processed_dataset.zip"),
    Path("/content/flatfoot_processed_dataset.zip")
]

target_zip = None
for candidate in zip_candidates:
    if candidate.exists():
        target_zip = candidate
        break

if target_zip is None:
    # Try mounting Google Drive
    try:
        from google.colab import drive
        print("Mounting Google Drive to look for dataset...")
        drive.mount('/content/drive')
        for candidate in zip_candidates:
            if candidate.exists():
                target_zip = candidate
                break
    except Exception as e:
        print("Google Drive mount skipped or failed:", e)

if target_zip and target_zip.exists():
    print(f"✓ Found dataset zip at: {target_zip}")
    print("Extracting dataset...")
    with zipfile.ZipFile(target_zip, 'r') as zf:
        zf.extractall(".")
    print("✓ Extraction complete!")
else:
    print("⚠️ Dataset zip not found automatically.")
    print("Please upload 'flatfoot_processed_dataset.zip' to Google Drive or directly to Colab via the files tab on the left.")

# Verify extraction
train_csv = Path("data/splits/train.csv")
val_csv = Path("data/splits/val.csv")
test_csv = Path("data/splits/test.csv")
processed_dir = Path("data/processed")

if train_csv.exists() and test_csv.exists() and processed_dir.exists():
    import pandas as pd
    train_df = pd.read_csv(train_csv)
    val_df = pd.read_csv(val_csv)
    test_df = pd.read_csv(test_csv)
    print(f"✓ Dataset verified successfully:")
    print(f"  - Training samples:   {len(train_df)} ({train_df['label'].value_counts().to_dict()})")
    print(f"  - Validation samples: {len(val_df)} ({val_df['label'].value_counts().to_dict()})")
    print(f"  - Test samples:       {len(test_df)} ({test_df['label'].value_counts().to_dict()})")
else:
    print("⚠️ Dataset paths not yet verified. Please ensure data is extracted to data/processed and data/splits.")



## 5. Instantiate Custom Model: FootArchNet
Inspect the novel architecture, feature taps, anisotropic Strip Pooling, and parameter profile.



In [ ]:
from src.models.foot_arch_net import create_foot_arch_net
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

# Instantiate custom FootArchNet
model = create_foot_arch_net(
    num_classes=2,
    pretrained=True,
    dropout=0.3,
    backbone_type="efficientnet_b2"
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 60)
print("FOOTARCHNET ARCHITECTURE SUMMARY")
print("=" * 60)
print(f"Total Parameters:     {total_params / 1e6:.2f} Million")
print(f"Trainable Parameters: {trainable_params / 1e6:.2f} Million")
print("Components:")
print("  - Stage Low (Cortical & Joint Spaces): Mid-level feature tap")
print("  - Stage High (Arch Morphology): High-level feature tap")
print("  - Strip Pooling (BSAM): Horizontal longitudinal arch + vertical height attention")
print("  - Biomechanical Attention: Spatial + Channel gating")
print("  - Dual Global Pooling: Concat(AvgPool, MaxPool) -> 768-dim embedding")
print("  - Classification Head: LayerNorm -> MLP -> GELU -> Dropout -> 2 Classes")
print("=" * 60)

# Quick forward test with dummy tensor
dummy_x = torch.randn(2, 3, 512, 512, device=device)
with torch.no_grad():
    dummy_out = model(dummy_x)
print(f"Forward pass sanity check: Input (2, 3, 512, 512) -> Output {dummy_out.shape} ✓")



## 6. Train FootArchNet with AMP & Cosine Annealing
Train for 20 epochs using class-weighted Cross-Entropy Loss to handle cohort distribution.



In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import pandas as pd
from pathlib import Path
from src.dataset import create_dataloaders
from src.training.trainer import Trainer

# Training hyperparameters
EPOCHS = 20
BATCH_SIZE = 8  # 8 is optimal for 512x512 resolution on T4/V100/3050Ti
LR = 1e-4
WEIGHT_DECAY = 1e-2
IMG_SIZE = 512

# Dataloaders
train_loader, val_loader, test_loader = create_dataloaders(
    data_dir="data",
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=2
)

# Class-weighted loss: 908 pes planus (class 1), 621 normal (class 0)
# w0 = 1529 / (2 * 621) = 1.231, w1 = 1529 / (2 * 908) = 0.842
class_weights = torch.tensor([1.231, 0.842], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

run_dir = Path("experiments/run_footarchnet_512px")
run_dir.mkdir(parents=True, exist_ok=True)

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    device=device,
    use_amp=True,
    output_dir=run_dir,
    patience=8
)

print(f"Starting FootArchNet training for {EPOCHS} epochs on {device}...")
history = trainer.fit(train_loader, val_loader, epochs=EPOCHS)
print("✓ Training finished!")



## 7. Comprehensive Test Evaluation (Held-Out Test Cohort: n=230)
Quantify diagnostic performance (Accuracy, Sensitivity, Specificity, F1, ROC-AUC) and compare directly against the standard ResNet-50 and EfficientNet-B2 benchmarks.



In [ ]:
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report
import numpy as np

# Load best checkpoint
best_checkpoint_path = run_dir / "best_model.pt"
checkpoint = torch.load(best_checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Run evaluation on test loader
all_preds, all_probs, all_targets, all_paths = [], [], [], []

with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device)
        with torch.amp.autocast('cuda' if device.type == 'cuda' else 'cpu'):
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())
        all_targets.extend(targets.numpy())

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_targets = np.array(all_targets)

# Compute Clinical Metrics
cm = confusion_matrix(all_targets, all_preds)
tn, fp, fn, tp = cm.ravel()

acc = (tp + tn) / (tp + tn + fp + fn)
sens = tp / (tp + fn)  # Sensitivity / Recall
spec = tn / (tn + fp)  # Specificity
prec = tp / (tp + fp)  # Precision / PPV
npv = tn / (tn + fn)   # Negative Predictive Value
f1 = 2 * (prec * sens) / (prec + sens)
fpr, tpr, _ = roc_curve(all_targets, all_probs)
roc_auc = auc(fpr, tpr)

metrics = {
    "model": "FootArchNet (Custom Multi-Scale Attention)",
    "accuracy": round(float(acc), 4),
    "sensitivity": round(float(sens), 4),
    "specificity": round(float(spec), 4),
    "precision": round(float(prec), 4),
    "npv": round(float(npv), 4),
    "f1_score": round(float(f1), 4),
    "roc_auc": round(float(roc_auc), 4)
}

print("=" * 65)
print("FOOTARCHNET TEST COHORT DIAGNOSTIC RESULTS (n=230)")
print("=" * 65)
print(f"Accuracy:            {acc*100:.2f}%")
print(f"Sensitivity (Recall): {sens*100:.2f}% (Flatfoot detected: {tp}/{tp+fn})")
print(f"Specificity:         {spec*100:.2f}% (Normal detected: {tn}/{tn+fp})")
print(f"Precision (PPV):     {prec*100:.2f}%")
print(f"NPV:                 {npv*100:.2f}%")
print(f"F1-Score:            {f1:.4f}")
print(f"ROC-AUC:             {roc_auc:.4f}")
print("=" * 65)

# Benchmark Comparison Table
comparison_data = {
    "Model Architecture": ["ResNet-50", "ConvNeXt-Tiny", "EfficientNet-B2", "FootArchNet (Ours)"],
    "Accuracy": ["84.35%", "84.35%", "86.09%", f"{acc*100:.2f}%"],
    "Sensitivity": ["94.89%", "94.16%", "91.24%", f"{sens*100:.2f}%"],
    "Specificity": ["68.82%", "69.89%", "78.49%", f"{spec*100:.2f}%"],
    "F1-Score": ["0.8784", "0.8776", "0.8865", f"{f1:.4f}"],
    "ROC-AUC": ["0.9134", "0.9152", "0.9222", f"{roc_auc:.4f}"]
}
comp_df = pd.DataFrame(comparison_data)
print("
BENCHMARK COMPARISON TABLE:")
display(comp_df)

# Plot Confusion Matrix and ROC Curve side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Normal", "Pes Planus"],
            yticklabels=["Normal", "Pes Planus"])
axes[0].set_title("FootArchNet Confusion Matrix", fontsize=12, fontweight="bold")
axes[0].set_ylabel("True Clinical Label")
axes[0].set_xlabel("Predicted Label")

# ROC Curve comparison
axes[1].plot(fpr, tpr, color="#059669", lw=2.5, label=f"FootArchNet (AUC = {roc_auc:.4f})")
# Plot baseline reference lines
axes[1].plot([0, 0.215, 1], [0, 0.9124, 1], color="#2563eb", lw=1.5, ls="--", label="EfficientNet-B2 (AUC = 0.9222)")
axes[1].plot([0, 0.312, 1], [0, 0.9489, 1], color="#7c3aed", lw=1.5, ls=":", label="ResNet-50 (AUC = 0.9134)")
axes[1].plot([0, 1], [0, 1], color="#94a3b8", lw=1, ls="--")
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel("False Positive Rate (1 - Specificity)")
axes[1].set_ylabel("True Positive Rate (Sensitivity)")
axes[1].set_title("ROC Comparison vs. Standard CNN Baselines", fontsize=12, fontweight="bold")
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(run_dir / "footarchnet_evaluation.png", dpi=300)
plt.show()



## 8. Grad-CAM Anatomical Explainability Validation
Visualize where FootArchNet focuses across clinical test cases to verify alignment with orthopaedic criteria (medial longitudinal arch, navicular-cuneiform drop, calcaneal pitch).



In [ ]:
from src.models.gradcam import GradCAM
import cv2
from PIL import Image

# Target layer is the Biomechanical Multi-Scale Fusion convolution block
target_layer = model.fusion_block.fusion_conv[-3]
gradcam = GradCAM(model, target_layer)

# Pick representative normal and pes planus samples from test predictions
test_df = pd.read_csv("data/splits/test.csv")
sample_indices = [0, 1, 2, 137, 138, 139]  # 3 Normal, 3 Pes Planus

fig, axes = plt.subplots(len(sample_indices), 2, figsize=(10, 3.2 * len(sample_indices)))
fig.suptitle("FootArchNet Grad-CAM Anatomical Saliency Maps", fontsize=14, y=0.995, fontweight="bold")

from src.dataset import get_transforms
eval_tf = get_transforms(512, is_training=False)

for row_idx, idx in enumerate(sample_indices):
    sample_row = test_df.iloc[idx]
    img_path = sample_row["path"]
    true_label = int(sample_row["label"])

    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (512, 512))

    tensor = eval_tf(Image.fromarray(img_rgb)).unsqueeze(0).to(device)
    cam = gradcam.generate(tensor, target_class=true_label)
    overlay = GradCAM.overlay_heatmap(img_resized, cam, alpha=0.45)

    with torch.no_grad():
        prob = torch.softmax(model(tensor), dim=1)[0, 1].item()
    pred_label = 1 if prob >= 0.5 else 0

    class_name = "Pes Planus" if true_label == 1 else "Normal Foot"
    pred_name = "Pes Planus" if pred_label == 1 else "Normal Foot"
    status = "CORRECT" if true_label == pred_label else "MISCLASSIFIED"

    axes[row_idx, 0].imshow(img_resized)
    axes[row_idx, 0].set_title(f"Original X-Ray: {class_name}", fontsize=10)
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(overlay)
    axes[row_idx, 1].set_title(f"FootArchNet Grad-CAM: {pred_name} (p={prob:.2f}) [{status}]", fontsize=10)
    axes[row_idx, 1].axis("off")

plt.tight_layout()
plt.savefig(run_dir / "footarchnet_gradcam_test.png", dpi=300)
plt.show()
print(f"✓ Grad-CAM saliency maps saved to {run_dir / 'footarchnet_gradcam_test.png'}")



## 9. Save Checkpoints & Results to Google Drive
Export model weights (`best_model.pt`) and evaluation charts directly to your Google Drive folder for permanent storage.



In [ ]:
import shutil

drive_export_dir = Path("/content/drive/MyDrive/flatfoot_classifier_results")
try:
    drive_export_dir.mkdir(parents=True, exist_ok=True)
    for f in run_dir.glob("*"):
        if f.is_file():
            shutil.copy(f, drive_export_dir / f.name)
    print(f"✓ All FootArchNet weights and figures successfully backed up to Google Drive at: {drive_export_dir}")
except Exception as e:
    print("Could not copy directly to Google Drive:", e)
    print("You can download the files directly from the Colab file browser on the left under experiments/run_footarchnet_512px/.")

